# Colorado reservoir storage — liberation pipeline

One notebook, four stages: **Retrieve → Audit → Cleanup → Publish-to-CSV**, thin
orchestration over the tested `reservoir` package (logic lives in `src/reservoir/`,
so it is unit-tested and reusable). Implements issue
[#9](https://github.com/CUPIDS-Lab/co-environmental-data/issues/9).

| Stage | Does | Writes |
|---|---|---|
| 1 · Retrieve | pull raw API responses (immutable) | `data/original/` + manifest |
| 2 · Audit | profile the retrieval | `data/audit/raw-profile-*.md` |
| 3 · Cleanup | parse → tidy long → validate | `data/processed/reservoir-storage.csv` + provenance |
| 4 · Publish | finalize CSV + audit + reconcile | `data/audit/summary-*.md`, `docs/variables.csv` |

**Run order:** top to bottom. `uv sync` first (or `pip install -e .`).

> **CDSS notes (confirmed live, 2026-06):** the value field is `measValue`; reservoir
> `abbrev`s are codes (Green Mountain → `GRERESCO`); CDSS returns **HTTP 404 for any
> zero-record query** — `fetch` treats that as *no data*, not an error, so one empty
> series never crashes the run.

In [ ]:
# Make the `reservoir` package importable (cleaner: `uv pip install -e ..`).
import sys, pathlib
SRC = pathlib.Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from reservoir import config, fetch, clean, audit
from IPython.display import Markdown
import pandas as pd, json
registry = config.get_sources(); list(registry)

## 1 · Retrieve  *(RETRIEVAL)*

`discover()` is offline — it builds the API request URLs without downloading.
Inspect the plan, then fetch. Originals land in `data/original/` and are immutable.

In [ ]:
plan = [{'source': s, 'reservoir': a.metadata.get('reservoir_name'), 'file': a.local_path.name}
        for s, src in registry.items() for a in src.discover()]
plan_df = pd.DataFrame(plan)
print(f'{len(plan_df)} artifacts planned'); plan_df.groupby('source').size()

**Run config** (set in the next cell):

- `MODE` — `"live"` fetches from the real APIs (**full history per site**); `"demo"` seeds one bundled fixture so stages 2–4 run with no network.
- `FRESH` — `True` clears `data/original/` first so the CSV reflects **exactly this run** (the originals are a rebuildable cache; clearing avoids stale partial data left by an earlier code version). Set `False` to reuse the cache once a full pull is done.
- `SOURCES` — `None` = all; or `["dwr_cdss"]` / `["reclamation_rise"]` to split the fast DWR pull from the large RISE one.

DWR/CDSS and RISE (resolved reservoirs) return data; Northern Water is boundaries-only (its C‑BT reservoirs are sourced from RISE). A full live pull is large — RISE histories paginate to ~1M rows — and streams progress below.

In [ ]:
# --- run configuration -------------------------------------------------------
MODE    = "live"    # "live" = fetch from the APIs | "demo" = offline 1-reservoir sample
FRESH   = True      # clear data/original first so the CSV reflects exactly THIS run
SOURCES = None      # None = all sources; or ["dwr_cdss"] / ["reclamation_rise"] to split

import shutil
if FRESH and config.ORIGINAL.exists():
    shutil.rmtree(config.ORIGINAL); print("cleared data/original (FRESH run)")
config.ORIGINAL.mkdir(parents=True, exist_ok=True)

if MODE == "live":
    # full history per site (DWR clamps to each station's record; RISE paginates).
    # progress streams below; RISE full pulls are large — set SOURCES to split if needed.
    fetch.fetch_all(sources=SOURCES)
else:
    fx = config.PROJECT_DIR / "tests/fixtures/dwr_cdss_storage_sample.json"
    art = next(a for a in registry["dwr_cdss"].discover()
               if a.metadata.get("reservoir_id") == "GRERESCO" and a.local_path.name.endswith("STORAGE.json"))
    art.local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(fx, art.local_path)
    print("offline demo seeded:", art.local_path.relative_to(config.PROJECT_DIR))

In [ ]:
files = sorted(p for p in config.ORIGINAL.rglob('*.json') if p.name != 'manifest.json')
print(f'{len(files)} raw files in data/original/')
assert files, 'No raw files — run the fetch cell (MODE live or demo).'
if (config.AUDIT/'fetch_errors.json').exists():
    print('fetch errors:', len(json.loads((config.AUDIT/'fetch_errors.json').read_text())))

## 2 · Audit  *(profile the retrieval)*

Before cleaning, profile what came back: which sources produced data, how many
files, what the response shapes are. Catches an empty/failed pull early.

In [ ]:
Markdown(audit.profile_raw())

## 3 · Cleanup  *(TIDY)*

Route each response through its parser → **tidy long** (one row per
`source · reservoir_id · datetime · variable`), harmonize units, preserve QA flags,
validate against the pandera schema, write the CSV + provenance sidecar. Errors are
durable (a failed parse logs to `extraction_errors.json` and the run continues).

In [ ]:
data, prov = clean.run(sources=SOURCES, fail_on_empty=False)
print("rows:", f"{len(data):,}", "| reservoirs:", data["reservoir_id"].nunique(),
      "| sources:", data.groupby("source")["reservoir_id"].nunique().to_dict())

# coverage per site — confirms FULL history landed for every reservoir (not just ~1 year)
from reservoir import audit
cov = audit.coverage_report()
print("span:", cov["first"].min().date(), "->", cov["last"].max().date(),
      "| median record:", f"{cov['years'].median():.0f}y",
      "| shortest:", f"{cov['years'].min():.0f}y | longest: {cov['years'].max():.0f}y")
data.head(8)

In [ ]:
prov

In [ ]:
errs = config.AUDIT / 'extraction_errors.json'
print(errs.read_text() if errs.exists() else 'no extraction errors ✓')

## 4 · Publish to CSV  *(PUBLISH)*

The tidy CSV is the deliverable. Finalize: full processed audit, the auto-generated
column profile (the data dictionary's sanity check), and reconciliation against each
agency's published totals. (Datasette/Quarto surfaces are deferred to the Hub roadmap.)

In [ ]:
Markdown(audit.audit_processed())

In [ ]:
audit.variables_report()

Fill `expected` with current storage off each agency's page (see `docs/survey-notes.md`); any mismatch beyond tolerance is a regression.

In [ ]:
expected = {
    # ('dwr_cdss', 'GRERESCO'): 57176.7,   # ← confirm from the DWR station page
}
audit.reconcile(expected) if expected else print('reconcile: fill expected totals (survey-notes)')

In [ ]:
csv = config.CANONICAL_CSV
print('deliverable:', csv.relative_to(config.PROJECT_DIR))
print('load: pd.read_csv(%r, parse_dates=[\'datetime\'])' % csv.name)
print('wide views -> docs/filter-pivot-recipes.md')